# Rain Prediction — Antecedent-moisture features (EXP-005)

EXP-004 showed temporal features add a small, consistent gain, and that the **strongest** of them
was `days_since_rain` (dry-spell length) — pointing at **antecedent moisture**, not yesterday's
instantaneous state, as the productive signal. This notebook tests that directly by adding:

- **Rolling rain sums** over 7 / 14 / 30 days,
- **Rain-day counts** over 7 / 14 / 30 days,
- an **Antecedent Precipitation Index** `API_t = k*API_{t-1} + P_{t-1}` (k=0.9) — the hydrology
  standard for decayed soil-moisture memory.

All are built from the **raw pre-MICE** series, strictly `<= t-1` (no same-day, no future, no
MICE-imputed future). A 3-way ablation — **base → base+temporal → base+temporal+antecedent** —
on **both** the chronological and leave-stations-out splits isolates the antecedent increment over
EXP-004's temporal set. Logs -> `ANTECEDENT_result.txt`, figures `antecedent_*`. Colab GPU.


## 1. Setup, imports, constants

In [ ]:
# --- Colab environment setup ---
import subprocess, sys as _sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([_sys.executable, '-m', 'pip', '-q', 'install', '-U', 'xgboost'], check=False)
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)
    # from google.colab import drive; drive.mount('/content/drive')
print(f'Running on Colab: {IN_COLAB}')

In [ ]:
import os, sys, zipfile, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    roc_curve, precision_recall_curve, auc,
)
import xgboost as xgb
from xgboost import XGBClassifier
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

RANDOM_STATE = 42
TEST_CUTOFF = pd.Timestamp('2016-01-01')

def _gpu_available():
    try:
        d = xgb.DMatrix(np.zeros((8, 2)), label=np.zeros(8))
        xgb.train({'device': 'cuda', 'tree_method': 'hist'}, d, num_boost_round=1)
        return True
    except Exception as e:
        print(f'CUDA probe failed ({type(e).__name__}); CPU fallback.'); return False

USE_GPU = _gpu_available()
DEVICE = 'cuda' if USE_GPU else 'cpu'
SEARCH_N_JOBS = 1 if USE_GPU else -1
XGB_KW = dict(tree_method='hist', device=DEVICE, eval_metric='logloss',
              random_state=RANDOM_STATE, n_jobs=-1)
def make_xgb(**ov):
    p = dict(XGB_KW); p.update(ov); return XGBClassifier(**p)

V2_PARAMS = dict(max_depth=6, min_child_weight=10, gamma=2.0, reg_lambda=20, reg_alpha=0.5,
                 learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_estimators=800)
print(f'xgboost {xgb.__version__} | device: {DEVICE}')

### Output setup

In [ ]:
os.makedirs('figures', exist_ok=True)
_PATH = 'ANTECEDENT_result.txt'
_FILE = open(_PATH, 'w', encoding='utf-8')
class _Tee:
    def __init__(self, *s): self._s = s
    def write(self, d):
        for s in self._s: s.write(d)
    def flush(self):
        for s in self._s: s.flush()
    def isatty(self): return False
_ORIG = sys.stdout; sys.stdout = _Tee(_ORIG, _FILE)
print('# Antecedent-moisture features (both splits) — numerical outputs')
print(f'# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
def log_section(t):
    print('\n' + '=' * 72 + '\n' + t + '\n' + '=' * 72)
def save_fig(name, fig=None):
    fig = fig or plt.gcf(); p = f'figures/{name}.png'
    fig.savefig(p, dpi=150, bbox_inches='tight'); print(f'  [saved {p}]')

## 2. Load raw (for lags/antecedent) + preprocessed (base features)

In [ ]:
log_section('2. Data loading')
def discover(c): return next((p for p in c if os.path.exists(p)), None)
RAW_PATH = discover(['weatherAUS.csv','data/weatherAUS.csv','../data/weatherAUS.csv',
    '/content/weatherAUS.csv','/content/drive/MyDrive/Colab_Notebooks/weatherAUS.csv'])
PRE_PATH = discover(['WeatherAUS_preprocessed_MICEcyclic.csv','data/WeatherAUS_preprocessed_MICEcyclic.csv',
    '../data/WeatherAUS_preprocessed_MICEcyclic.csv','/content/WeatherAUS_preprocessed_MICEcyclic.csv',
    '/content/drive/MyDrive/Colab_Notebooks/WeatherAUS_preprocessed_MICEcyclic.csv'])
if RAW_PATH is None or PRE_PATH is None:
    raise FileNotFoundError(f'raw={RAW_PATH}, pre={PRE_PATH}')
raw = pd.read_csv(RAW_PATH); raw['Date'] = pd.to_datetime(raw['Date'])
df  = pd.read_csv(PRE_PATH); df['Date'] = pd.to_datetime(df['Date'])
df['RainTomorrow'] = df['RainTomorrow'].astype(bool)
print(f'raw {raw.shape} | preprocessed {df.shape}')

## 3. Base features (v2)

In [ ]:
log_section('3. Base features (v2)')
if 'Date_sin' not in df.columns:
    d = df['Date'].dt.dayofyear
    df['Date_sin'] = np.sin(2*np.pi*d/365.25); df['Date_cos'] = np.cos(2*np.pi*d/365.25)
df['Pressure_diff'] = df['Pressure3pm'] - df['Pressure9am']
df['Temp_diff']     = df['Temp3pm']     - df['Temp9am']
BASE_FEATURES = ['MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine','WindGustSpeed',
    'WindSpeed9am','WindSpeed3pm','Humidity3pm','Pressure9am','Pressure3pm','Cloud9am','Cloud3pm',
    'Temp9am','Temp3pm','Date_sin','Date_cos','WindGustDir_sin','WindGustDir_cos','WindDir9am_sin',
    'WindDir9am_cos','WindDir3pm_sin','WindDir3pm_cos','Pressure_diff','Temp_diff']
print(f'BASE features: {len(BASE_FEATURES)}')

## 4. Temporal + antecedent features (raw, strictly backward)

Temporal block = the EXP-004 set (held identical for a clean ablation). Antecedent block = new
rolling rain windows, rain-day counts, and the decayed API. Everything uses only days `<= t-1`.

In [ ]:
log_section('4. Temporal + antecedent features (raw, backward)')
raw_s = raw.sort_values(['Location', 'Date']).copy()
g = raw_s.groupby('Location', sort=False)
feat = pd.DataFrame({'Date': raw_s['Date'].values, 'Location': raw_s['Location'].values})

# --- Temporal block (identical to EXP-004) ---
for c in ['Humidity3pm','Pressure3pm','Rainfall','Cloud3pm','Sunshine','Temp3pm','WindGustSpeed']:
    feat[f'{c}_lag1'] = g[c].shift(1).values
feat['Pressure_3d_chg']  = (g['Pressure3pm'].shift(1) - g['Pressure3pm'].shift(4)).values
feat['Humidity_3d_mean'] = g['Humidity3pm'].apply(lambda s: s.shift(1).rolling(3).mean()).values
feat['Rainfall_3d_sum']  = g['Rainfall'].apply(lambda s: s.shift(1).rolling(3).sum()).values
rt = raw_s['RainToday'].map({'Yes': 1, 'No': 0})
rt_lag = rt.groupby(raw_s['Location'], sort=False).shift(1)
feat['RainToday_lag1'] = rt_lag.values
def _days_since(s):
    out = np.empty(len(s)); cnt = 9999.0
    for i, v in enumerate(s.to_numpy()):
        out[i] = cnt
        cnt = 0.0 if v == 1 else (cnt + 1 if cnt < 9999 else 9999)
    return out
feat['days_since_rain'] = (
    rt_lag.fillna(0).groupby(raw_s['Location'], sort=False).transform(lambda s: _days_since(s)).values)
TEMPORAL_FEATURES = [c for c in feat.columns if c not in ('Date', 'Location')]

# --- Antecedent block (NEW) ---
rain = raw_s['Rainfall']
gb_rain = rain.groupby(raw_s['Location'], sort=False)
gb_rday = rt.groupby(raw_s['Location'], sort=False)
for w in (7, 14, 30):
    feat[f'Rainfall_{w}d_sum'] = gb_rain.transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum()).values
    feat[f'RainDays_{w}d']     = gb_rday.transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum()).values
def _api(p, k=0.9):
    out = np.empty(len(p)); a = 0.0
    for i, v in enumerate(p):
        out[i] = a                                  # API through t-1 (decayed past rainfall)
        a = k * a + (0.0 if np.isnan(v) else float(v))
    return out
feat['API_0p9'] = gb_rain.transform(lambda s: _api(s.to_numpy())).values
ANTECEDENT_FEATURES = ['Rainfall_7d_sum','Rainfall_14d_sum','Rainfall_30d_sum',
                       'RainDays_7d','RainDays_14d','RainDays_30d','API_0p9']

df = df.merge(feat, on=['Date', 'Location'], how='left')
print(f'Temporal features  : {len(TEMPORAL_FEATURES)}')
print(f'Antecedent features: {len(ANTECEDENT_FEATURES)} -> {ANTECEDENT_FEATURES}')
print('NaN share  temporal %.1f%%  antecedent %.1f%%' % (
    df[TEMPORAL_FEATURES].isna().mean().mean()*100, df[ANTECEDENT_FEATURES].isna().mean().mean()*100))

## 5. Two evaluation protocols + 3 feature sets

In [ ]:
log_section('5. Splits + feature sets')
df_sorted = df.sort_values('Date').reset_index(drop=True)
ALL_FEATURES = BASE_FEATURES + TEMPORAL_FEATURES + ANTECEDENT_FEATURES
X_all  = df_sorted[ALL_FEATURES]
y_all  = df_sorted['RainTomorrow'].astype(int)
loc_all = df_sorted['Location']
dates   = df_sorted['Date']

chrono_train = (dates < TEST_CUTOFF).values
print(f'Chronological: train {chrono_train.sum():,} | test {(~chrono_train).sum():,}')

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
sp_tr, sp_te = next(gss.split(df_sorted, y_all, groups=loc_all))
spatial_train = np.zeros(len(df_sorted), bool); spatial_train[sp_tr] = True
held = sorted(loc_all.iloc[sp_te].unique())
print(f'Leave-stations-out: {len(held)} held-out -> {held}')

SPLITS = {'chronological': dict(train=chrono_train, cv='time'),
          'leave-stations-out': dict(train=spatial_train, cv='group')}
FEATURE_SETS = {
    'base':                      BASE_FEATURES,
    'base+temporal':             BASE_FEATURES + TEMPORAL_FEATURES,
    'base+temporal+antecedent':  ALL_FEATURES,
}

## 6. Helpers — PR-crossover threshold + evaluation

In [ ]:
def pr_crossover_threshold(params, Xtr, ytr, cv, groups, grid=np.arange(0.10, 0.81, 0.01)):
    recs = []
    for tr, va in cv.split(Xtr, ytr, groups):
        m = make_xgb(**params); m.fit(Xtr.iloc[tr], ytr.iloc[tr])
        recs.append((ytr.iloc[va].values, m.predict_proba(Xtr.iloc[va])[:, 1]))
    best_t, best_d = 0.5, 1e9
    for t in grid:
        P = np.mean([precision_score(y, p >= t, zero_division=0) for y, p in recs])
        R = np.mean([recall_score(y, p >= t) for y, p in recs])
        F = np.mean([f1_score(y, p >= t) for y, p in recs])
        d = abs(P - F) + abs(F - R)
        if d < best_d: best_t, best_d = float(t), d
    return round(best_t, 2)

def evaluate(fs_name, cols, split_name, train_mask, cv_kind):
    Xtr = X_all.loc[train_mask, cols].reset_index(drop=True)
    Xte = X_all.loc[~train_mask, cols].reset_index(drop=True)
    ytr = y_all[train_mask].reset_index(drop=True)
    yte = y_all[~train_mask].reset_index(drop=True)
    pr = float(ytr.mean()); params = dict(V2_PARAMS); params['scale_pos_weight'] = (1 - pr) / pr
    if cv_kind == 'time':
        cv, groups = TimeSeriesSplit(n_splits=5), None
    else:
        cv, groups = GroupKFold(n_splits=5), loc_all[train_mask].reset_index(drop=True)
    t = pr_crossover_threshold(params, Xtr, ytr, cv, groups)
    m = make_xgb(**params); m.fit(Xtr, ytr)
    proba = m.predict_proba(Xte)[:, 1]; pred = (proba >= t).astype(int)
    p, r, _ = precision_recall_curve(yte, proba)
    row = dict(split=split_name, features=fs_name, n_feat=len(cols), threshold=t,
               F1=round(f1_score(yte, pred), 4), precision=round(precision_score(yte, pred), 4),
               recall=round(recall_score(yte, pred), 4),
               ROC_AUC=round(roc_auc_score(yte, proba), 4), PR_AUC=round(auc(r, p), 4))
    return m, row, proba, yte

## 7. Run 2 splits x 3 feature sets

In [ ]:
log_section('7. Results — 2 splits x 3 feature sets')
results, store = [], {}
for sp_name, sp in SPLITS.items():
    for fs_name, cols in FEATURE_SETS.items():
        m, row, proba, yte = evaluate(fs_name, cols, sp_name, sp['train'], sp['cv'])
        results.append(row); store[(sp_name, fs_name)] = (m, proba, yte)
        print(f"  {sp_name:20s} {fs_name:26s} -> F1 {row['F1']:.4f}  ROC {row['ROC_AUC']:.4f}  PR {row['PR_AUC']:.4f}")
res = pd.DataFrame(results)
print('\nMaster table:')
print(res.to_string(index=False))

## 8. ROC / PR curves per split (3 feature sets)

In [ ]:
for sp_name in SPLITS:
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for fs_name in FEATURE_SETS:
        _, proba, yte = store[(sp_name, fs_name)]
        fpr, tpr, _ = roc_curve(yte, proba)
        ax[0].plot(fpr, tpr, label=f'{fs_name} ({roc_auc_score(yte, proba):.3f})')
        p, r, _ = precision_recall_curve(yte, proba)
        ax[1].plot(r, p, label=f'{fs_name} ({auc(r, p):.3f})')
    ax[0].plot([0, 1], [0, 1], 'k--', alpha=0.4); ax[0].set_title(f'ROC — {sp_name}')
    ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend(loc='lower right', fontsize=8); ax[0].grid(alpha=0.3)
    ax[1].axhline(float(store[(sp_name, "base")][2].mean()), color='gray', ls=':', label='base rate')
    ax[1].set_title(f'PR — {sp_name}'); ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
    ax[1].legend(loc='upper right', fontsize=8); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    save_fig(f'antecedent_roc_pr_{sp_name.replace("-", "_")}', fig); plt.show()

## 9. Feature importance — antecedent (red) vs temporal (orange) vs base (blue)

In [ ]:
log_section('9. Importance (chronological, full feature set)')
m_full = store[('chronological', 'base+temporal+antecedent')][0]
gain = m_full.get_booster().get_score(importance_type='gain')
def kind(f):
    if f in ANTECEDENT_FEATURES: return 'antecedent'
    if f in TEMPORAL_FEATURES:   return 'temporal'
    return 'base'
imp = pd.DataFrame([(k, gain.get(k, 0.0), kind(k)) for k in ALL_FEATURES],
                   columns=['feature', 'gain', 'kind']).sort_values('gain', ascending=False).reset_index(drop=True)
print('Top 20 by gain (A=antecedent, T=temporal):')
for _, r in imp.head(20).iterrows():
    tag = {'antecedent': 'A', 'temporal': 'T', 'base': ' '}[r.kind]
    print(f'  {tag} {r.feature:20s} {r.gain:8.2f}')
print('\nAntecedent features and their ranks:')
for f in ANTECEDENT_FEATURES:
    rank = int(imp.index[imp.feature == f][0]) + 1
    print(f'  {f:18s} rank {rank}/{len(imp)}  gain {float(imp.loc[imp.feature==f,"gain"].iloc[0]):.2f}')

cmap = {'antecedent': 'crimson', 'temporal': 'darkorange', 'base': 'steelblue'}
fig, ax = plt.subplots(figsize=(9, 7))
top = imp.head(20)
ax.barh(top['feature'][::-1], top['gain'][::-1], color=[cmap[k] for k in top['kind']][::-1])
ax.set_xlabel('gain'); ax.set_title('Top-20 gain — antecedent (red) / temporal (orange) / base (blue)')
ax.grid(axis='x', ls='--', alpha=0.5); plt.tight_layout()
save_fig('antecedent_feature_importance', fig); plt.show()

## 10. Verdict — does antecedent moisture add over temporal?

In [ ]:
log_section('10. Verdict')
def row(sp, fs): return res[(res.split == sp) & (res.features == fs)].iloc[0]
for sp in SPLITS:
    b  = row(sp, 'base'); t = row(sp, 'base+temporal'); a = row(sp, 'base+temporal+antecedent')
    print(f'{sp}:')
    print(f'  base -> +temporal     dF1 {t.F1-b.F1:+.4f} | dROC {t.ROC_AUC-b.ROC_AUC:+.4f} | dPR {t.PR_AUC-b.PR_AUC:+.4f}')
    print(f'  +temporal -> +anteced  dF1 {a.F1-t.F1:+.4f} | dROC {a.ROC_AUC-t.ROC_AUC:+.4f} | dPR {a.PR_AUC-t.PR_AUC:+.4f}')
    print(f'  base -> +all (total)   dF1 {a.F1-b.F1:+.4f} | dROC {a.ROC_AUC-b.ROC_AUC:+.4f} | dPR {a.PR_AUC-b.PR_AUC:+.4f}')

c_t, c_a = row('chronological', 'base+temporal'), row('chronological', 'base+temporal+antecedent')
d_ante = max(c_a.F1 - c_t.F1, c_a.PR_AUC - c_t.PR_AUC, c_a.ROC_AUC - c_t.ROC_AUC)
print()
if d_ante >= 0.003:
    print('VERDICT: antecedent-moisture features ADD over the temporal set on the forecasting split.')
    print('  -> keep them; the productive axis is multi-day moisture memory. Try API decay sweep / longer windows.')
else:
    print('VERDICT: antecedent features do NOT add materially over the temporal set.')
    print('  -> days_since_rain already captured the dry-spell signal; pivot to spatial context or target reframe.')

## Finalization

In [ ]:
log_section('Finalization')
print(f'Results file: {_PATH}')
figs = sorted(f for f in os.listdir('figures') if f.startswith('antecedent_'))
for f in figs: print(f'  figures/{f}')
sys.stdout = _ORIG; _FILE.close()
ZIP = 'XGB_antecedent_artifacts.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(_PATH)
    for f in figs: zf.write(f'figures/{f}')
print(f'Bundled -> {ZIP}')
if IN_COLAB:
    from google.colab import files; files.download(ZIP)
print(f'Done. See {_PATH} and figures/.')